# Решения: set/dict частоты

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


CSV_PATH = _find('orders_slim.csv')
df = pd.read_csv(
    CSV_PATH,
    parse_dates=['order_purchase_timestamp', 'order_estimated_delivery_date', 'order_delivered_customer_date'],
)


In [ ]:
unique_orders = set(df['order_id'])
freq: dict[str, dict[str, int]] = {}
for r in df[['customer_state', 'is_late']].itertuples(index=False):
    st = r.customer_state
    if st not in freq:
        freq[st] = {'late': 0, 'total': 0}
    freq[st]['total'] += 1
    freq[st]['late'] += int(r.is_late)
rate = {k: v['late'] / v['total'] for k, v in freq.items()}
seller_count: dict[str, int] = {}
for r in df[['seller_id', 'is_late']].itertuples(index=False):
    if r.is_late == 1:
        seller_count[r.seller_id] = seller_count.get(r.seller_id, 0) + 1
bad_states = {k for k, v in rate.items() if v > 0.6}
top_late = dict(sorted(seller_count.items(), key=lambda x: x[1], reverse=True)[:5])
print('unique:', len(unique_orders))
print('freq:', freq)
print('rate:', {k: round(v, 3) for k, v in rate.items()})
print('bad_states:', bad_states)
print('top_late:', top_late)